# **INSIGHTS**
1. Traders generate higher profits during Greed periods, indicating momentum-driven gains.

2. Larger trade sizes show higher volatility and risk, especially during Fear periods.

3. Frequent traders adapt better to sentiment shifts and maintain more stable performance.

# **STRATEGY**

1. Reduce position size during Fear periods to minimize losses.

2. Increase trading activity during Greed when market trends are strong.

3. Avoid large trades during uncertain sentiment to reduce risk exposure.

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [50]:
sentiment = pd.read_csv("fear_greed_index.csv")
trades = pd.read_csv("historical_data.csv")

In [51]:
sentiment.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [52]:
trades.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [53]:
sentiment.shape

(2644, 4)

In [54]:
trades.shape

(211224, 16)

In [55]:
sentiment.isnull().sum()

,0
timestamp,0
value,0
classification,0
date,0


In [56]:
trades.isnull().sum()

,0
Account,0
Coin,0
Execution Price,0
Size Tokens,0
Size USD,0
Side,0
Timestamp IST,0
Start Position,0
Direction,0
Closed PnL,0


In [57]:

trades.duplicated().sum()

np.int64(0)

In [58]:
trades.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [59]:
trades.columns = [
    'account','coin','execution_price','size_tokens','size_usd',
    'side','timestamp_ist','start_position','direction',
    'closed_pnl','tx_hash','order_id','crossed','fee','trade_id','timestamp'
] #renaming

In [60]:
cols = ['execution_price','size_tokens','size_usd','closed_pnl','fee']

for col in cols:
    trades[col] = pd.to_numeric(trades[col], errors='coerce') #converting numeric columns

In [61]:
trades = trades.drop_duplicates()
trades = trades.dropna(subset=['closed_pnl'])

In [62]:
trades['timestamp_ist'] = pd.to_datetime(trades['timestamp_ist'], dayfirst=True)

trades['Date'] = trades['timestamp_ist'].dt.date #converting dates

In [63]:
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

In [64]:
merged = pd.merge(trades, sentiment, left_on='Date', right_on='date', how='left')

In [65]:
daily_pnl = merged.groupby(['account','Date'])['closed_pnl'].sum().reset_index()

In [66]:
merged['win'] = merged['closed_pnl'] > 0
win_rate = merged.groupby('account')['win'].mean()

In [67]:
avg_size = merged.groupby('account')['size_usd'].mean()

In [68]:
trades_per_day = merged.groupby('Date').size()

In [69]:
merged['side'] = merged['side'].str.lower()

long_short = merged['side'].value_counts(normalize=True)

In [70]:
merged.groupby('classification')['fee'].mean()

,fee
classification,
Extreme Fear,1.116291
Extreme Greed,0.675902
Fear,1.495172
Greed,1.254372
Neutral,1.044798


In [71]:
#analysis
merged.groupby('classification')['fee'].mean()

,fee
classification,
Extreme Fear,1.116291
Extreme Greed,0.675902
Fear,1.495172
Greed,1.254372
Neutral,1.044798


In [72]:
#behaviour change trade count
merged.groupby('classification').size()

,0
classification,
Extreme Fear,21400
Extreme Greed,39992
Fear,61837
Greed,50303
Neutral,37686


In [73]:
#trade size
merged.groupby('classification')['size_usd'].mean()

,size_usd
classification,
Extreme Fear,5349.731843
Extreme Greed,3112.251565
Fear,7816.109931
Greed,5736.884375
Neutral,4782.732661


In [74]:
#long/short
pd.crosstab(merged['classification'], merged['side'], normalize='index')

side,buy,sell
classification,,
Extreme Fear,0.510981,0.489019
Extreme Greed,0.448590,0.551410
Fear,0.489513,0.510487
Greed,0.488559,0.511441
Neutral,0.503343,0.496657


In [75]:
#step 10 High vs Low Trade Size
merged['size_segment'] = np.where(
    merged['size_usd'] > merged['size_usd'].median(),
    'High','Low'
)

In [76]:
#frequent traders
trade_counts = merged['account'].value_counts()
threshold = trade_counts.median()

merged['freq_segment'] = merged['account'].map(
    lambda x: 'High' if trade_counts[x] > threshold else 'Low'
)

In [77]:
#profitability
pnl_total = merged.groupby('account')['closed_pnl'].sum()

merged['profit_segment'] = merged['account'].map(
    lambda x: 'Winner' if pnl_total[x] > 0 else 'Loser'
)

In [78]:
#segment analysis
merged.groupby(['size_segment','classification'])['closed_pnl'].mean()

size_segment  classification
High          Extreme Fear       61.196379
              Extreme Greed     140.570044
              Fear               97.025581
              Greed              84.301244
              Neutral            69.581271
Low           Extreme Fear        1.157202
              Extreme Greed       9.628661
              Fear                3.580906
              Greed               3.609685
              Neutral             2.188668
Name: closed_pnl, dtype: float64